<a href="https://colab.research.google.com/github/chetools/CHE4061_Spring2026/blob/main/MESHDistillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget -N -q https://raw.githubusercontent.com/chetools/chetools/main/tools/che5.ipynb -O che5.ipynb
%run che5.ipynb

In [2]:
import numpy as np
import jax
import jax.numpy as jnp
jax.config.update("jax_enable_x64", True)
from scipy.optimize import root_scalar, minimize_scalar, bracket
from scipy.optimize import root
from scipy.special import expit, logit
from plotly.subplots import make_subplots
from scipy.interpolate import Akima1DInterpolator
np.set_printoptions(precision=5)

In [3]:
R=8.314
p=Props(['Methanol', 'Ethanol', 'Isopropanol', 'Water'])

In [4]:
def dewP_ideal(y, T):
    P=1./(np.sum(y/p.Pvap(T)))
    return P, y*P/p.Pvap(T)

def dewT_ideal(y, P):

    def P_dev(T):
        return dewP_ideal(y, T)[0] - P
    T = root(P_dev, 300.).x[0]

    return T,  y*P/p.Pvap(T)

In [5]:
def bubbleP_NRTL(x, T):
    Pi= x*p.NRTL_gamma(x,T)*p.Pvap(T)
    P=np.sum(Pi)
    return P, Pi/P

def bubbleT_NRTL(x, P):

    def f(T):
        return bubbleP_NRTL(x,T)[0]-P

    #mole-fraction weighted boiling points of each component at P
    #boiling points determined via Clausius Clapeyron, using the Hvap at the normal bp
    #for each component.  p.Hvap returns the heat of vaporization of all components for each
    #temperature if an array of temperatures is given.
    Tguess=np.dot(x,1/(1/p.Tbn-np.log(P/101325)*R/np.diagonal(p.Hvap(p.Tbn))))

    res=root_scalar(f, x0=Tguess, method='secant')
    if not(res.converged):
        return "FAIL", res
    T=root_scalar(f, x0=Tguess, method='secant').root
    Pi= x*p.NRTL_gamma(x,T)*p.Pvap(T)
    P=np.sum(Pi)

    return T, Pi/P



In [6]:
def dewP_NRTL(y, T):
    def f(vec):
        P = vec[0]
        x = expit(vec[1:])  #Ensures that mole fractions are between 0 and 1

        fug_eqs = x*p.NRTL_gamma(x,T) * p.Pvap(T)  - y*P
        xsum_eq = 1. - np.sum(x)

        return np.r_[fug_eqs, xsum_eq]

    #Assume ideal liquid dewP calculation for initial guess of P and liquid phase composition
    Pguess, xguess= dewP_ideal(y,T)

    #f (function to zero) maps values from -inf to inf, to values between 0 and 1
    #so xguess is mapped via logit which is the inverse function of expit
    v0 = np.r_[Pguess, logit(xguess)]
    res=root(f, v0)
    if not(res.success):
        return "FAILURE", res
    return res.x[0], expit(res.x[1:])

In [7]:
def dewT_NRTL(y, P):
    def f(vec):
        T = vec[0]
        x = expit(vec[1:])  #Ensures that mole fractions are between 0 and 1

        fug_eqs = x*p.NRTL_gamma(x,T) * p.Pvap(T)  - y*P
        xsum_eq = 1. - np.sum(x)

        return np.r_[fug_eqs, xsum_eq]

    #Assume ideal liquid dewP calculation for initial guess of P and liquid phase composition
    Tguess, xguess= dewT_ideal(y,P)

    #f (function to zero) maps values from -inf to inf, to values between 0 and 1
    #so xguess is mapped via logit which is the inverse function of expit
    v0 = np.r_[Tguess, logit(xguess)]
    res=root(f, v0)
    if not(res.success):
        return "FAILURE", res
    return res.x[0], expit(res.x[1:])

In [8]:
def flash_idealPT(z, P, T):

    K=p.Pvap(T)/P
    def rachford(VF):
        return np.sum(z*(K-1)/(VF*(K-1) +1))

    res=root_scalar(rachford, bracket=(0,1))
    VF = res.root
    x=z/(1-VF + K *VF)
    y=K*x
    return x, y, VF

In [9]:
def flash_NRTL_PT(z, P, T, maxiter = 100, tol=1e-12):

    dewP, dewx = dewP_NRTL(z, T)
    bubbleP, bubbley = bubbleP_NRTL(z,T)

    xguess = (P-dewP)/(bubbleP-dewP) * (z - dewx) +  dewx

    for i in range(maxiter):
        K=p.NRTL_gamma(xguess,T)*p.Pvap(T)/P

        def rachford(VF):
            return np.sum(z*(K-1)/(VF*(K-1) +1))

        res=root_scalar(rachford, bracket=(0,1))
        VF = res.root
        x=z/(1-VF + K *VF)
        if (np.linalg.norm(xguess-x)<tol):
            break
        xguess = x

    y=K*x
    return x, y, VF, i

In [10]:
def flash_NRTL_PT(z, P, T, maxiter = 100, tol=1e-12):

    dewP, dewx = dewP_NRTL(z, T)
    bubbleP, bubbley = bubbleP_NRTL(z,T)

    xguess = (P-dewP)/(bubbleP-dewP) * (z - dewx) +  dewx

    for i in range(maxiter):
        K=p.NRTL_gamma(xguess,T)*p.Pvap(T)/P

        def rachford(VF):
            return np.sum(z*(K-1)/(VF*(K-1) +1))

        res=root_scalar(rachford, bracket=(0,1))
        VF = res.root
        x=z/(1-VF + K *VF)
        if (np.linalg.norm(xguess-x)<tol):
            break
        xguess = x

    y=K*x
    return x, y, VF, i



In [58]:
Nc = p.Mw.size
Ftot = 1.
z = np.array([0.25,0.25,0.25, 0.25])
P = 4e4
dewT, _ = dewT_NRTL(z, P)
bubbleT, _ = bubbleT_NRTL(z,P)
feedT = (bubbleT + dewT)/2
x,y, vf, _ = flash_NRTL_PT(z, P, feedT)
feedH=p.Hv(Ftot*vf*y, feedT) + p.Hl(Ftot*(1-vf)*x,feedT)

In [57]:
bubbleP_NRTL(jnp.array([0.5, 0.5, 0., 0.]), 320)

(Array(37056.39003, dtype=float64),
 Array([0.65465, 0.34535, 0.     , 0.     ], dtype=float64))

In [59]:
D = 0.5*Ftot
Btot = Ftot - D
R = 10.
Ns = 40
Nf = 20

unk = np.zeros((Ns, 2*Nc+1))
Ltot_rec = R*D
Ltot_strip = Ltot_rec + Ftot*(1-vf)
Vtot_rec = R*D + D
Vtot_strip = Vtot_rec - Ftot*vf
unk[:Nf,:Nc] = Ltot_rec*y
unk[Nf:-1,:Nc] = Ltot_strip*x
unk[-1,:Nc] = Btot*x
unk[:Nf,Nc:2*Nc ] = Vtot_rec*y
unk[Nf:,Nc:2*Nc ] = Vtot_strip*x
unk[:,-1]=np.linspace(bubbleT_NRTL(y,P)[0],dewT_NRTL(x,P)[0],Ns)

In [60]:
def stage1(vec, vec2, refluxT):
    T,T2 = vec[-1], vec2[-1]
    L,V = jnp.split(vec[:-1],2)
    L2,V2 = jnp.split(vec2[:-1],2)
    x = L/jnp.sum(L)
    y = V/jnp.sum(V)

    MB = (V2 + R*D*y - V - L)/Ftot
    EQ = x*p.NRTL_gamma(x,T)*p.Pvap(T)/P - y
    EB = (p.Hv(V2, T2) + p.Hl(R*D*y, refluxT) - p.Hv(V, T) - p.Hl(L, T))/feedH

    return jnp.r_[MB, EQ, EB]

def stage(vec1, vec, vec2, f, fH):
    T1, T,T2 = vec1[-1], vec[-1], vec2[-1]
    L1,V1 = jnp.split(vec1[:-1],2)
    L,V = jnp.split(vec[:-1],2)
    L2,V2 = jnp.split(vec2[:-1],2)
    x = L/jnp.sum(L)
    y = V/jnp.sum(V)

    MB = (f + V2 + L1 - V - L)/Ftot
    EQ = x*p.NRTL_gamma(x,T)*p.Pvap(T)/P - y
    EB = (fH + p.Hv(V2, T2) + p.Hl(L1, T1) - p.Hv(V, T) - p.Hl(L, T))/feedH

    return jnp.r_[MB, EQ, EB]


def stageN(vec1, vec):
    T1, T = vec1[-1], vec[-1]
    L1,V1 = jnp.split(vec1[:-1],2)
    L,V = jnp.split(vec[:-1],2)
    x = L/jnp.sum(L)
    y = V/jnp.sum(V)

    MB = (L1 - V - L)/Ftot
    EQ = x*p.NRTL_gamma(x,T)*p.Pvap(T)/P - y
    MB2 = (jnp.sum(L) - Btot)/Btot

    return jnp.r_[MB, EQ, MB2]

In [61]:
stage1_jac = jax.jit(jax.jacobian(stage1, (0,1)))
stage_jac = jax.jit(jax.jacobian(stage, (0,1,2)))
stageN_jac = jax.jit(jax.jacobian(stageN, (0,1)))

In [62]:
Es = np.zeros((Ns, 2*Nc+1))
Cs = np.zeros((Ns-1, 2*Nc+1, 2*Nc+1))

In [63]:
def evalEs(unk):

    L,V = np.split(unk[0,:-1],2)
    y=V/np.sum(V)
    refluxT = bubbleT_NRTL(y,P)[0]
    Es[0] = stage1(unk[0], unk[1], refluxT)
    for i in range(1, Ns-1):
        if i==Nf:
            Es[i]= stage(unk[i-1], unk[i], unk[i+1], Ftot*z, feedH)
        else:
            Es[i]= stage(unk[i-1], unk[i], unk[i+1], 0., 0.)

    Es[-1] = stageN(unk[-2], unk[-1])

    return Es

def norm_evalEs(unk):
    return np.linalg.norm(evalEs(unk))


In [64]:
for iter in range(1,25):
    Es= evalEs(unk)
    normEs = norm_evalEs(unk)
    print(iter, normEs)
    if normEs<1e-8:
        break
    L,V = np.split(unk[0,:-1],2)
    y=V/np.sum(V)
    refluxT = bubbleT_NRTL(y,P)[0]
    B,C = stage1_jac(unk[0], unk[1], refluxT)
    Binv = np.linalg.inv(B)
    Cs[0]= Binv @ C
    Es[0]= Binv @ Es[0]
    for i in range(1, Ns-1):
        if i==Nf:
            A,B,C = stage_jac(unk[i-1], unk[i], unk[i+1], Ftot*z, feedH)
        else:
            A,B,C= stage_jac(unk[i-1], unk[i], unk[i+1], 0., 0.)

        Binv = np.linalg.inv(B-A@Cs[i-1])
        Cs[i]=Binv@C
        Es[i]=Binv@(Es[i]-A@Es[i-1])

    A,B = stageN_jac(unk[-2], unk[-1])
    Binv = np.linalg.inv(B-A@Cs[-1])
    Es[-1] = Binv@ (Es[-1] - A@Es[-2])

    delta_unk = np.zeros_like(unk)
    delta_unk[-1]= Es[-1]
    for i in range(Ns-2,-1,-1):
        delta_unk[i]= Es[i]-Cs[i] @ delta_unk[i+1]

    brac = bracket(lambda t: norm_evalEs(unk + t*delta_unk), xa=0., xb=1.)
    t = minimize_scalar(lambda t: norm_evalEs(unk + t*delta_unk), brac[:3]).x
    unk = unk + t*delta_unk

1 1.545796926913041
2 0.2104241943651646
3 0.006524794034165713
4 8.383370721888638e-05
5 9.558338119600536e-08
6 9.73135558070984e-11


In [65]:

L,V = np.split(unk[:,:-1],2, axis=1)
x=L/np.sum(L, axis=1)[:,None]
y=V/np.sum(V, axis=1)[:,None]

In [66]:
x

array([[3.46380e-01, 4.51223e-01, 9.78658e-02, 1.04531e-01],
       [2.34971e-01, 4.99968e-01, 1.31456e-01, 1.33605e-01],
       [1.63548e-01, 5.19507e-01, 1.60976e-01, 1.55969e-01],
       [1.20878e-01, 5.20309e-01, 1.86349e-01, 1.72465e-01],
       [9.63528e-02, 5.10554e-01, 2.08368e-01, 1.84725e-01],
       [8.25824e-02, 4.95413e-01, 2.27845e-01, 1.94160e-01],
       [7.49937e-02, 4.77882e-01, 2.45378e-01, 2.01746e-01],
       [7.09037e-02, 4.59639e-01, 2.61352e-01, 2.08106e-01],
       [6.87771e-02, 4.41605e-01, 2.75998e-01, 2.13620e-01],
       [6.77451e-02, 4.24276e-01, 2.89451e-01, 2.18528e-01],
       [6.73191e-02, 4.07907e-01, 3.01789e-01, 2.22986e-01],
       [6.72252e-02, 3.92613e-01, 3.13054e-01, 2.27108e-01],
       [6.73103e-02, 3.78430e-01, 3.23268e-01, 2.30992e-01],
       [6.74897e-02, 3.65339e-01, 3.32436e-01, 2.34736e-01],
       [6.77172e-02, 3.53287e-01, 3.40541e-01, 2.38455e-01],
       [6.79693e-02, 3.42196e-01, 3.47541e-01, 2.42294e-01],
       [6.82355e-02, 3.3

In [33]:
y

array([[3.84512e-01, 3.31435e-01, 1.65831e-01, 1.18222e-01],
       [2.79949e-01, 3.60898e-01, 2.11013e-01, 1.48140e-01],
       [2.09027e-01, 3.72608e-01, 2.47353e-01, 1.71012e-01],
       [1.63757e-01, 3.73188e-01, 2.75541e-01, 1.87514e-01],
       [1.35750e-01, 3.67653e-01, 2.97390e-01, 1.99207e-01],
       [1.18707e-01, 3.59134e-01, 3.14606e-01, 2.07553e-01],
       [1.08442e-01, 3.49435e-01, 3.28477e-01, 2.13645e-01],
       [1.02316e-01, 3.39546e-01, 3.39905e-01, 2.18233e-01],
       [9.86987e-02, 3.29991e-01, 3.49496e-01, 2.21814e-01],
       [9.65980e-02, 3.21026e-01, 3.57661e-01, 2.24714e-01],
       [9.54103e-02, 3.12762e-01, 3.64678e-01, 2.27150e-01],
       [9.47700e-02, 3.05223e-01, 3.70734e-01, 2.29273e-01],
       [9.44562e-02, 2.98390e-01, 3.75958e-01, 2.31196e-01],
       [9.43361e-02, 2.92216e-01, 3.80431e-01, 2.33016e-01],
       [9.43303e-02, 2.86645e-01, 3.84202e-01, 2.34823e-01],
       [9.43921e-02, 2.81607e-01, 3.87283e-01, 2.36718e-01],
       [9.44955e-02, 2.7

In [ ]:
opx = np.r_[y[0], np.repeat(x[:-1],2), x[-1]]
opx

array([0.41762, 0.33703, 0.24535, 0.39643, 0.39643, 0.352  , 0.352  ,
       0.25156, 0.25156, 0.37911, 0.37911, 0.36366, 0.36366, 0.25723,
       0.25723, 0.36491, 0.36491, 0.37245, 0.37245, 0.26264, 0.26264,
       0.35321, 0.35321, 0.37869, 0.37869, 0.26809, 0.26809, 0.34345,
       0.34345, 0.38261, 0.38261, 0.27394, 0.27394, 0.33512, 0.33512,
       0.38428, 0.38428, 0.2806 , 0.2806 , 0.32776, 0.32776, 0.38361,
       0.38361, 0.28863, 0.28863, 0.32086, 0.32086, 0.38028, 0.38028,
       0.29886, 0.29886, 0.3138 , 0.3138 , 0.3736 , 0.3736 , 0.3126 ,
       0.3126 , 0.30563, 0.30563, 0.36219, 0.36219, 0.33217, 0.33217,
       0.29455, 0.29455, 0.34318, 0.34318, 0.36226, 0.36226, 0.29064,
       0.29064, 0.34636, 0.34636, 0.36299, 0.36299, 0.28622, 0.28622,
       0.34988, 0.34988, 0.36389, 0.36389, 0.28121, 0.28121, 0.3537 ,
       0.3537 , 0.36509, 0.36509, 0.27545, 0.27545, 0.35767, 0.35767,
       0.36688, 0.36688, 0.26866, 0.26866, 0.36132, 0.36132, 0.37002,
       0.37002, 0.26

In [ ]:
opy=np.r_[np.repeat(y,2)]

In [ ]:
P=101325
x1s=np.linspace(0,1,101)
Ts=[]
y1s=[]
for x1 in x1s:
    T, (y1,y2) = bubbleT_NRTL(np.array([x1, 1-x1]), P)
    Ts.append(T)
    y1s.append(y1)

In [ ]:
fig = make_subplots()
fig.add_scatter(x=x1s, y=y1s)
fig.add_scatter(x=opx, y=opy)
fig.update_layout(width=600, height=600, template='plotly_dark')
